## PROTÓTIPO MAIS BÁSICO PARA EXIBIÇÃO 


In [35]:
from rdkit import Chem
from rdkit.Chem import Draw

In [36]:
# Armazenando a string SMILES em uma variável
smiles_input = "CC(O)C(=O)O"

In [37]:
# Criando um objeto Mol a partir da string SMILES (conversão de SMILES para Mol)
molecula = Chem.MolFromSmiles(smiles_input)

In [38]:
# Armazenando a imagem da molécula em uma variável
imagem = Draw.MolToImage(molecula)


In [39]:
# Exibindo em 2d
imagem.show()

In [40]:
# Para exibir em 3d precisamos da biblioteca py3Dmol
import py3Dmol


In [41]:
# Estrutura básica para exibir a molécula em 3D

view = py3Dmol.view(width=400, height=300) # Cria um visualizador 3D com largura de 400 pixels e altura de 300 pixels
view.addModel(Chem.MolToMolBlock(molecula), "sdf") # Adiciona o modelo da molécula ao visualizador
view.setStyle({"stick": {}}) # Define o estilo de visualização como stick
view.zoomTo() # Ajusta o zoom para a molécula
view.show() # Exibe a molécula em 3D

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [42]:
# Estrutura mais avançada para exibir a molécula em 3D com diferentes estilos de visualização
advanced_view = py3Dmol.view(width=400, height=300) # Cria um visualizador 3D com largura de 400 pixels e altura de 300 pixels
advanced_view.addModel(Chem.MolToMolBlock(molecula), "sdf")
advanced_view.setStyle({"stick": {}, "sphere": {"scale": 0.3}}) # Define o estilo de visualização como stick
advanced_view.zoomTo() # Ajusta o zoom para a molécula
advanced_view.show() # Exibe a molécula em 3D com estilos avançados


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## TESTANDO ALGUMAS POSSIBILIDADES

#### CONTAGEM DE ÁTOMOS DE CADA ELEMENTO.

In [43]:
# vamos precisar do rdMolDescriptors para calcular as propriedades moleculares
from rdkit.Chem import rdMolDescriptors

In [44]:
# Como já instanciamos a molécula, podemos calcular algumas propriedades com a variável molecula.
formula = rdMolDescriptors.CalcMolFormula(molecula)
print(f"Fórmula molecular: {formula}")

Fórmula molecular: C3H6O3


### Para treinar modelos de machine learning, precisamos fazer a contagem detalhada, pois os hidrogênio são implícitos.


In [ ]:
# vamos ter que importar Counter 
from collections import Counter # é uma classe que permite contar a ocorrência de elementos em um iterável, como uma lista ou uma string.

# já temos a molecula em uma variável, então podemos usar a função GetAtoms() para obter os átomos da molécula e depois contar a ocorrência de cada átomo usando Counter.

mol_com_hidrogenios = Chem.AddHs(molecula) # Adiciona hidrogênios explícitos à molécula


In [ ]:
# contagem (iterar sobre os átomos da molécula e contar a ocorrência de cada elemento)
contagem_elementos = Counter(
    [atom.GetSymbol() for atom in mol_com_hidrogenios.GetAtoms()]
)

# este código cria um dicionário com a contagem de cada elemento químico na molécula, incluindo os hidrogênios explícitos. Por exemplo, para a molécula de ácido lático (CC(O)C(=O)O), a saída será algo como:

In [34]:
print(dict(contagem_elementos))  # dict é usado para converter o objeto Counter em um dicionário padrão do Python, facilitando a visualização da contagem de elementos.

{'C': 3, 'O': 3, 'H': 6}


## VAMOS TENTAR MELHORAR A VIZUALIZAÇÃO 3D
#### Precisamos preparar a molécula com H e coordenaas 3d reais. Vamos usar três exemplos com carbono em diferentes hibridizações: sp3, sp2 e sp. Vamos usar o RDKit para gerar as coordenadas 3D e depois exibir as moléculas em 3D usando py3Dmol.


In [47]:
# Para isso usamos AllChem que é um módulo do RDKit que contém funções para química computacional, incluindo geração de coordenadas 3D e otimização de geometria.
from rdkit.Chem import AllChem

In [49]:

# CH4 hibridização sp3

smiles_C_sp3 = "C"
mol_C_sp3 = Chem.MolFromSmiles(smiles_C_sp3)
mol_C_sp3 = Chem.AddHs(mol_C_sp3)  # Adiciona H
AllChem.EmbedMolecule(mol_C_sp3)  # Gera coordenadas 3D
AllChem.MMFFOptimizeMolecule(mol_C_sp3)  # Otimiza a geometria


0

In [50]:
# Precisamos convertar para bloco SDF/MOL pois o py3Dmol não aceita o objeto Mol diretamente, ele precisa de uma representação em string do modelo molecular. O bloco SDF/MOL é um formato padrão para representar moléculas, incluindo informações sobre átomos, ligações e coordenadas 3D.

mol_block_C_sp3 = Chem.MolToMolBlock(mol_C_sp3)

In [63]:
# Agora sim, vamos exibir o Modelo de Esferas de Van der Waals de forma propocional

view_ballstick_c_sp3 = py3Dmol.view(width=450, height=350)
view_ballstick_c_sp3.addModel(mol_block_C_sp3, "sdf")


# Define bastões (stick) + esferas proporcionais (sphere) com esquema de cores CPK / Jmol
view_ballstick_c_sp3.setStyle({
    "stick": {"radius": 0.14},
    "sphere": {"scale": 0.28, "colorscheme": "Jmol"}
})

view_ballstick_c_sp3.zoomTo()
view_ballstick_c_sp3.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Tentando cirar uma Superfície Aproximada (Incrustação de Densidade/Superfície Molecular)

#### Para representar a nuvem/superfície da molécula (colorida por cargas elétricas parciais de Gasteiger), o RDKit calcula a distribuição de carga local e o py3Dmol renderiza uma malha semi-transparente por cima dos átomos:

In [81]:
# 1. Preparar a molécula com hidrogênios e coordenadas 3D
mol_mep = Chem.MolFromSmiles("O")  # Exemplo com a água (H2O) ou troque pelo seu SMILES
mol_mep = Chem.AddHs(mol_mep)
AllChem.EmbedMolecule(mol_mep)
AllChem.MMFFOptimizeMolecule(mol_mep)

# 2. Calcular as cargas parciais de Gasteiger no RDKit
AllChem.ComputeGasteigerCharges(mol_mep)
mol_block = Chem.MolToMolBlock(mol_mep)

# 3. Renderizar com o py3Dmol aplicando o gradiente de cores por carga
view = py3Dmol.view(width=500, height=400)
view.addModel(mol_block, "sdf")

# Esqueleto interno em Ball & Stick
view.setStyle({
    "stick": {"radius": 0.12},
    "sphere": {"scale": 0.25, "colorscheme": "Jmol"}

})

# Superfície translúcida com mapa de cores (Vermelho = Negativo, Branco = Neutro, Azul = Positivo)
view.addSurface(
    py3Dmol.VDW,
    {
        "opacity": 0.7,
        "colorscheme": {
            "prop": "partialcharge",
            "gradient": "rwb",  # Red - White - Blue
            "min": -0.4,
            "max": 0.4
        }
    }
)

view.zoomTo()
view.show()



3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## VAMOS TENTAR AGORA, EXTRAÇÃO DE PROPRIEDADES.

### Descritores Físico-Químicos Diretos


In [86]:

from rdkit.Chem import Descriptors

# Reutilizando a variável 'molecula' (Ácido Lático) do notebook
massa = Descriptors.MolWt(molecula)
logp = Descriptors.MolLogP(molecula)
tpsa = Descriptors.TPSA(molecula)

print(f"Massa Molecular: {massa:.2f} g/mol")
print(f"LogP (Lipofilicidade): {logp:.2f}")
print(f"TPSA (Área de Superfície Polar): {tpsa:.2f} Å²")

Massa Molecular: 90.08 g/mol
LogP (Lipofilicidade): -0.55
TPSA (Área de Superfície Polar): 57.53 Å²


### Contagem de Doadores e Aceitadores de Hidrogênio

In [85]:
doadores_h = Descriptors.NumHDonors(molecula)
aceitadores_h = Descriptors.NumHAcceptors(molecula)
ligacoes_rot = Descriptors.NumRotatableBonds(molecula)

print(f"Doadores de Ligação de H: {doadores_h}")
print(f"Aceitadores de Ligação de H: {aceitadores_h}")
print(f"Ligações Rotacionáveis: {ligacoes_rot}")

Doadores de Ligação de H: 2
Aceitadores de Ligação de H: 2
Ligações Rotacionáveis: 1


### Detecção de Grupos Funcionais (SMARTS Matching)

In [87]:
# Definindo padrões SMARTS para Ácido Carboxílico e Álcool
grupo_carboxila = Chem.MolFromSmarts("C(=O)[OH]")
grupo_hidroxila = Chem.MolFromSmarts("[NX3,NX4;H2,H1][CX4]") # Exemplo para teste ou ajuste

tem_carboxila = molecula.HasSubstructMatch(grupo_carboxila)
print(f"Possui grupo Ácido Carboxílico? {tem_carboxila}")

Possui grupo Ácido Carboxílico? True


### Validação da Regra de Lipinski (Fármaco-similaridade)
#### avalia se uma molécula tem perfil de candidato a fármaco oral (Regra dos 5 de Lipinski)

In [88]:
# Validação direta dos parâmetros da Regra de Lipinski
lipinski = {
    "peso_ok": Descriptors.MolWt(molecula) <= 500,
    "logp_ok": Descriptors.MolLogP(molecula) <= 5,
    "doadores_ok": Descriptors.NumHDonors(molecula) <= 5,
    "aceitadores_ok": Descriptors.NumHAcceptors(molecula) <= 10
}

aprovado_lipinski = all(lipinski.values())
print(f"Aprovado na Regra de Lipinski? {aprovado_lipinski}")
print(f"Detalhes: {lipinski}")

Aprovado na Regra de Lipinski? True
Detalhes: {'peso_ok': True, 'logp_ok': True, 'doadores_ok': True, 'aceitadores_ok': True}


### Identificação de Centros Quirais e Estereoquímica

In [ ]:
# Localizando centros quirais na 'molecula'
centros_quirais = Chem.FindMolChiralCenters(molecula, includeUnassigned=True)
print(f"Centros Quirais identificados (Índice do Átomo, Tipo): {centros_quirais}")



Centros Quirais identificados (Índice do Átomo, Tipo): [(1, '?')]


In [90]:
# SMILES com estereoquímica definida (L-ácido lático)
mol_estereo = Chem.MolFromSmiles("C[C@@H](O)C(=O)O")

# Como a estereoquímica já vem no SMILES, precisamos apenas chamar AssignStereochemistry
Chem.AssignStereochemistry(mol_estereo)

centros = Chem.FindMolChiralCenters(mol_estereo)
print(f"Centro Quiral e Configuração: {centros}")

Centro Quiral e Configuração: [(1, 'R')]
